# PCL Detection — RoBERTa-base + Multi-task Auxiliary Loss + Threshold Optimisation

**Approach:**
- **RoBERTa-base** fine-tuned for binary PCL classification
- **Class-weighted `BCEWithLogitsLoss`** (`pos_weight = n_neg/n_pos ≈ 9.55`) — single correction for class imbalance (no sampler double-correction)
- **Multi-task auxiliary loss** (weight = 0.3): model outputs 8 logits — logit[0] is binary PCL, logits[1–7] are the 7 PCL sub-categories. Auxiliary supervision forces encoder to learn fine-grained PCL framing
- **Threshold optimisation** on dev set (search [0.05, 0.95])
- **Two-stage**: train on train → pick best epoch & threshold → retrain on train+dev → predict test

Baseline: 0.48 dev / 0.49 test.

In [ ]:
import os, ast, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
CFG = {
    'model_name':   'roberta-base',
    'main_data':    'dontpatronizeme_pcl.tsv',
    'train_ids':    'train_semeval_parids-labels.csv',
    'dev_ids':      'dev_semeval_parids-labels.csv',
    'test_data':    'task4_test.tsv',
    'max_length':   256,
    'batch_size':   16,
    'grad_accum':   2,
    'lr':           2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'num_epochs':   8,
    'patience':     3,
    'aux_weight':   0.3,   # weight for 7-category auxiliary loss
    'seed':         42,
    'best_model':   'BestModel/best_model_roberta.pt',
    'final_model':  'BestModel/final_model_roberta.pt',
    'dev_preds':    'BestModel/dev.txt',
    'test_preds':   'BestModel/test.txt',
}

PCL_CATEGORIES = [
    'Unbalanced power relations', 'Shallow solution', 'Presupposition',
    'Authority voice', 'Metaphor', 'Compassion', 'The poorer the merrier',
]

os.makedirs('BestModel', exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
set_seed(CFG['seed'])
print("Config ready.")

In [ ]:
def load_main_data(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t', 5)
            if len(parts) == 6 and parts[0].strip().isdigit():
                rows.append({'par_id': int(parts[0]), 'keyword': parts[2], 'country': parts[3],
                             'text': parts[4].strip(), 'label_raw': int(parts[5])})
    df = pd.DataFrame(rows)
    df['label_binary'] = (df['label_raw'] >= 2).astype(int)
    return df

def load_split_ids(path):
    df = pd.read_csv(path)
    df['par_id'] = df['par_id'].astype(int)
    df['label_list'] = df['label'].apply(ast.literal_eval)
    return df[['par_id', 'label_list']]

def load_test_data(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) >= 5 and parts[0].strip().startswith('t_'):
                rows.append({'par_id': parts[0].strip(), 'text': parts[4].strip() if len(parts) > 4 else ''})
    df = pd.DataFrame(rows)
    return df[df['text'].str.len() > 0].reset_index(drop=True)

df_main      = load_main_data(CFG['main_data'])
df_train_ids = load_split_ids(CFG['train_ids'])
df_dev_ids   = load_split_ids(CFG['dev_ids'])
df_test      = load_test_data(CFG['test_data'])

df_train = df_main[df_main['par_id'].isin(df_train_ids['par_id'])].copy()
df_dev   = df_main[df_main['par_id'].isin(df_dev_ids['par_id'])].copy()

df_train = df_train.merge(df_train_ids, on='par_id', how='left').reset_index(drop=True)
df_dev   = df_dev.merge(df_dev_ids,   on='par_id', how='left').reset_index(drop=True)

_zero7 = lambda x: x if isinstance(x, list) else [0] * 7
df_train['label_list'] = df_train['label_list'].apply(_zero7)
df_dev['label_list']   = df_dev['label_list'].apply(_zero7)

n_pos = (df_train['label_binary'] == 1).sum()
n_neg = (df_train['label_binary'] == 0).sum()
print(f"Train: {len(df_train)}, Dev: {len(df_dev)}, Test: {len(df_test)}")
print(f"Train PCL: {n_pos}, No-PCL: {n_neg}  ratio {n_neg/n_pos:.2f}:1")

In [ ]:
class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length, aux_labels=None):
        self.texts      = [str(t) for t in texts]
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.aux_labels = aux_labels   # list of 7-element lists, or None for test set

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_length,
                             padding='max_length', truncation=True, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(float(self.labels[idx]), dtype=torch.float)
        if self.aux_labels is not None:
            item['aux_labels'] = torch.tensor(self.aux_labels[idx], dtype=torch.float)
        return item

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])

train_ds = PCLDataset(df_train['text'].tolist(), df_train['label_binary'].tolist(),
                      tokenizer, CFG['max_length'], aux_labels=df_train['label_list'].tolist())
dev_ds   = PCLDataset(df_dev['text'].tolist(),   df_dev['label_binary'].tolist(),
                      tokenizer, CFG['max_length'], aux_labels=df_dev['label_list'].tolist())
test_ds  = PCLDataset(df_test['text'].tolist(),  [0]*len(df_test),
                      tokenizer, CFG['max_length'])  # no aux labels for test

# No WeightedRandomSampler — pos_weight alone corrects for class imbalance.
# Using both sampler + pos_weight double-corrects, which over-suppresses recall.
train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=0, pin_memory=True)
dev_dl   = DataLoader(dev_ds,   batch_size=CFG['batch_size']*2, shuffle=False, num_workers=0)
test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size']*2, shuffle=False, num_workers=0)

# Primary: class-weighted BCE (single correction for imbalance)
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Primary pos_weight = {pos_weight.item():.2f}")

# Auxiliary: per-label class weights for 7 PCL categories
train_arr  = np.array(df_train['label_list'].tolist(), dtype=np.float32)
pos_counts = train_arr.sum(axis=0).clip(min=1)
neg_counts = len(train_arr) - pos_counts
pw_aux     = torch.tensor(neg_counts / pos_counts, dtype=torch.float).to(device)
aux_criterion = nn.BCEWithLogitsLoss(pos_weight=pw_aux)
print("Auxiliary per-category pos_weights:")
for cat, w in zip(PCL_CATEGORIES, pw_aux.cpu().numpy()):
    print(f"  {cat:<35s}: {w:.1f}")

In [ ]:
# num_labels=8: logits[:, 0] = binary PCL, logits[:, 1:] = 7 PCL sub-categories

def run_epoch(model, loader, optimizer, scheduler, criterion, grad_accum,
              aux_criterion=None, aux_weight=0.3):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc='Train', leave=False)):
        logits = model(batch['input_ids'].to(device),
                       attention_mask=batch['attention_mask'].to(device)).logits
        loss = criterion(logits[:, 0], batch['labels'].to(device))
        if aux_criterion is not None and 'aux_labels' in batch:
            loss = loss + aux_weight * aux_criterion(logits[:, 1:], batch['aux_labels'].to(device))
        loss = loss / grad_accum
        loss.backward()
        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item() * grad_accum
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_logits, all_labels = [], []
    for batch in tqdm(loader, desc='Eval', leave=False):
        logits = model(batch['input_ids'].to(device),
                       attention_mask=batch['attention_mask'].to(device)).logits
        all_logits.append(logits[:, 0].cpu().numpy())  # binary logit only
        all_labels.append(batch['labels'].numpy())
    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    probs = 1.0 / (1.0 + np.exp(-all_logits))
    preds = (probs >= threshold).astype(int)
    return f1_score(all_labels.astype(int), preds, zero_division=0), probs, all_labels

def tune_threshold(probs, labels, t_min=0.05, t_max=0.95):
    best_t, best_f = 0.5, 0.0
    for t in np.linspace(t_min, t_max, 91):
        f = f1_score(labels.astype(int), (probs >= t).astype(int), zero_division=0)
        if f > best_f: best_f, best_t = f, t
    print(f"Best threshold: {best_t:.3f}  →  dev F1 = {best_f:.4f}")
    return float(best_t)

@torch.no_grad()
def predict(model, loader, threshold):
    model.eval()
    probs = []
    for batch in tqdm(loader, desc='Predict', leave=False):
        logits = model(batch['input_ids'].to(device),
                       attention_mask=batch['attention_mask'].to(device)).logits
        probs.append(torch.sigmoid(logits[:, 0]).cpu().numpy())
    probs = np.concatenate(probs)
    return (probs >= threshold).astype(int), probs

print("Helpers defined.")

In [ ]:
# Stage 1: train on train set, early stop on dev F1
set_seed(CFG['seed'])
model = AutoModelForSequenceClassification.from_pretrained(CFG['model_name'], num_labels=8).to(device)
total_steps = (len(train_dl) // CFG['grad_accum']) * CFG['num_epochs']
optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps*CFG['warmup_ratio']), total_steps)
print(f"Model: {CFG['model_name']}  |  num_labels=8  |  total steps: {total_steps}")

best_f1, best_epoch, best_probs, best_labels = 0.0, 0, None, None
patience_cnt = 0

for epoch in range(CFG['num_epochs']):
    print(f"\n── Epoch {epoch+1}/{CFG['num_epochs']} ──")
    train_loss = run_epoch(model, train_dl, optimizer, scheduler, criterion, CFG['grad_accum'],
                           aux_criterion=aux_criterion, aux_weight=CFG['aux_weight'])
    f1, probs, labels = evaluate(model, dev_dl, threshold=0.5)
    print(f"   Train loss: {train_loss:.4f}  |  Dev F1 (t=0.5): {f1:.4f}", end='')
    if f1 > best_f1:
        best_f1, best_epoch, best_probs, best_labels = f1, epoch+1, probs.copy(), labels.copy()
        patience_cnt = 0
        torch.save(model.state_dict(), CFG['best_model'])
        print("  ✓ best")
    else:
        patience_cnt += 1
        print(f"  [patience {patience_cnt}/{CFG['patience']}]")
        if patience_cnt >= CFG['patience']:
            print("Early stop.")
            break

print(f"\nBest dev F1 = {best_f1:.4f} at epoch {best_epoch}")

In [ ]:
# Threshold optimisation on dev
model.load_state_dict(torch.load(CFG['best_model'], map_location=device))
_, best_probs, best_labels = evaluate(model, dev_dl, threshold=0.5)
best_thresh = tune_threshold(best_probs, best_labels)
preds_dev = (best_probs >= best_thresh).astype(int)
print(classification_report(best_labels.astype(int), preds_dev, target_names=['No-PCL', 'PCL'], digits=4))
with open(CFG['dev_preds'], 'w') as f:
    for p in preds_dev: f.write(f"{int(p)}\n")
print(f"Saved {CFG['dev_preds']}")

In [ ]:
# Stage 2: retrain on train+dev for best_epoch epochs, then predict test
df_full  = pd.concat([df_train, df_dev], ignore_index=True)
n_pos_f  = (df_full['label_binary'] == 1).sum()
n_neg_f  = (df_full['label_binary'] == 0).sum()

full_ds  = PCLDataset(df_full['text'].tolist(), df_full['label_binary'].tolist(),
                      tokenizer, CFG['max_length'], aux_labels=df_full['label_list'].tolist())
full_dl  = DataLoader(full_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=0, pin_memory=True)

pos_weight_f    = torch.tensor([n_neg_f / n_pos_f], dtype=torch.float).to(device)
criterion_f     = nn.BCEWithLogitsLoss(pos_weight=pos_weight_f)

full_arr        = np.array(df_full['label_list'].tolist(), dtype=np.float32)
pw_aux_f        = torch.tensor((len(full_arr) - full_arr.sum(0)) / full_arr.sum(0).clip(1),
                               dtype=torch.float).to(device)
aux_criterion_f = nn.BCEWithLogitsLoss(pos_weight=pw_aux_f)

set_seed(CFG['seed'])
model_final = AutoModelForSequenceClassification.from_pretrained(CFG['model_name'], num_labels=8).to(device)
steps_s2    = (len(full_dl) // CFG['grad_accum']) * best_epoch
opt_s2      = AdamW(model_final.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched_s2    = get_linear_schedule_with_warmup(opt_s2, int(steps_s2*CFG['warmup_ratio']), steps_s2)
print(f"Stage 2: retraining for {best_epoch} epoch(s) on {len(df_full):,} samples (train+dev)")

for epoch in range(best_epoch):
    print(f"Stage 2 epoch {epoch+1}/{best_epoch}")
    run_epoch(model_final, full_dl, opt_s2, sched_s2, criterion_f, CFG['grad_accum'],
              aux_criterion=aux_criterion_f, aux_weight=CFG['aux_weight'])

torch.save(model_final.state_dict(), CFG['final_model'])

preds_test, probs_test = predict(model_final, test_dl, threshold=best_thresh)
np.save('BestModel/test_probs.npy', probs_test)   # save raw probs for diagnostics

with open(CFG['test_preds'], 'w') as f:
    for p in preds_test: f.write(f"{int(p)}\n")
print(f"Saved {CFG['test_preds']}  (threshold={best_thresh:.3f})")
print(f"Predicted PCL: {preds_test.sum()} / {len(preds_test)}  ({preds_test.mean()*100:.1f}%)")
print("\nBestModel/ contents:")
for fn in sorted(os.listdir('BestModel')):
    print(f"  {fn}")